In [21]:
import pandas as pd
import networkx as nx
import ast
from networkx.algorithms import bipartite

Primero se genera el grafo bipartito con 2 conjuntos, personas y episodios. Un link entre una persona y episodio existe solo si esta trabajo en la produccion del capitulo.

In [22]:
# Cargar CSV
df = pd.read_csv("datasets/one_piece_merged.csv")

# Columnas donde hay listas de personas
staff_columns = ["guion", "arte", "animacion", "direccion"]
value_columns = ["n de paginas", "rating", "votos"]

# Función para convertir string de lista a lista real
def parse_list(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

# Crear grafo bipartito
B = nx.Graph()

for _, row in df.iterrows():
    episodio = f"ep_{int(row['episodio'])}"
    
    # Agregar nodo episodio
    B.add_node(episodio, bipartite="episode")

    value_dict = {}
    for val in value_columns:
        value_dict[val] = row[val]
    
    nx.set_node_attributes(B, {episodio: value_dict})

    # Iterar por todas las áreas del staff
    for col in staff_columns:
        personas = parse_list(row[col])
        
        for persona in personas:
            persona_clean = persona.split("(")[0].strip()

            # Agregar nodo persona
            if persona_clean == '':
                continue
            B.add_node(persona_clean, bipartite="staff")

            # Agregar arista
            B.add_edge(persona_clean, episodio)


print(f"Number of nodes: {B.number_of_nodes()}, Number of edges: {B.number_of_edges()}")
nx.write_gexf(B, "one_piece.gexf")

Number of nodes: 1385, Number of edges: 5181


Se procede a sacar la proyeccion del grafo, haciendo que solo queden las personas del staff.

In [23]:
# Obtener nodos de staff
staff_nodes = {n for n, d in B.nodes(data=True) if d["bipartite"] == "staff"}

# Proyección
G_staff = bipartite.weighted_projected_graph(B, staff_nodes)

print(f"Number of nodes: {G_staff.number_of_nodes()}, Number of edges: {G_staff.number_of_edges()}")
nx.write_gexf(G_staff, "one_piece_staff_projection.gexf")

Number of nodes: 234, Number of edges: 2878


Centralidad de grado

In [24]:
degree = dict(G_staff.degree())

top = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]

for name, deg in top:
    print(name, deg)

Kazuya Hisada 110
Jin Tanaka 108
Kenji Yokoyama 101
Miyuki Sato 86
Atsuhiro Tomioka 85
Masayuki Takagi 85
Miho Shiraishi 80
Eisaku Inoue 79
Masahiro Shimanuki 78
Shōji Yonemura 77


Centralidad de grado normalizada

In [25]:
degree = nx.degree_centrality(G_staff)

top = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:10]

for name, deg in top:
    print(name, deg)

Kazuya Hisada 0.4721030042918455
Jin Tanaka 0.463519313304721
Kenji Yokoyama 0.4334763948497854
Miyuki Sato 0.36909871244635195
Atsuhiro Tomioka 0.3648068669527897
Masayuki Takagi 0.3648068669527897
Miho Shiraishi 0.34334763948497854
Eisaku Inoue 0.33905579399141633
Masahiro Shimanuki 0.33476394849785407
Shōji Yonemura 0.33047210300429186


Eigenvector centrality sin considerar peso de la arista

In [26]:
eigen = nx.eigenvector_centrality(G_staff)

top_eigen = sorted(eigen.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_eigen:
    print(name, val)

Kazuya Hisada 0.1853056667758415
Kenji Yokoyama 0.18212953002554008
Jin Tanaka 0.17417102977117177
Masahiro Shimanuki 0.15853813049190466
Toshio Deguchi 0.15422138838955032
Atsuhiro Tomioka 0.15204234815368922
Masayuki Takagi 0.15113270826558206
Miho Shiraishi 0.14623984844398807
Eisaku Inoue 0.14470702298212185
Tomohiro Nakayama 0.14433643451369557


Eigenvector centrality considerando el peso de la arista

In [27]:
eigen = nx.eigenvector_centrality(G_staff, weight='weight')

top_eigen = sorted(eigen.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_eigen:
    print(name, val)

Miyuki Sato 0.40429844337311527
Hirohiko Kamisaka 0.3083661688362665
Miho Shiraishi 0.30807424281980966
Ryūji Yoshiike 0.2724313422107723
Yoshiyuki Suga 0.2457162028563987
Kenji Yokoyama 0.2180676238359565
Masayuki Takagi 0.19880187226203186
Jin Tanaka 0.1967998848620237
Yoshihiro Ueda 0.1856794152011002
Michiyo Kawasaki 0.1764612076221813


Closeness

In [28]:
closeness = nx.closeness_centrality(G_staff)

top_closeness = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_closeness:
    print(name, val)

Kazuya Hisada 0.6544943820224719
Jin Tanaka 0.6508379888268156
Kenji Yokoyama 0.6383561643835617
Masayuki Takagi 0.6115485564304461
Eisaku Inoue 0.6005154639175257
Masahiro Shimanuki 0.6005154639175257
Miho Shiraishi 0.5974358974358974
Atsuhiro Tomioka 0.5943877551020408
Miyuki Sato 0.5913705583756346
Toshio Deguchi 0.5913705583756346


In [29]:
betweenness = nx.betweenness_centrality(G_staff)

top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

for name, val in top_betweenness:
    print(name, val)

Jin Tanaka 0.09633378873682828
Kazuya Hisada 0.09227593670101197
Miyuki Sato 0.06822647240279531
Kenji Yokoyama 0.05688206306731107
Eisaku Inoue 0.03927594467125317
Masayuki Takagi 0.037859808088085596
Masahiro Kitazaki 0.03661422610620598
Atsuhiro Tomioka 0.035113169319852706
Masahiro Shimanuki 0.029312730237190637
Hirohiko Kamisaka 0.029090178627029388
